In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import cv2
import os
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import keras
import pathlib
import cv2

HEIGHT = 224
WIDTH = 224
FRAMES_PER_VIDEO = 10
OVERLAP = 0

class DeepfakeVideoDetector:
    def __init__(self, num_frames=FRAMES_PER_VIDEO, img_size=(HEIGHT, WIDTH)):
        self.num_frames = num_frames
        self.img_size = img_size
        self.model = self.build_model()
    
    def build_model(self):
        # Входной слой для последовательности кадров
        input_layer = layers.Input(shape=(self.num_frames, self.img_size[0], self.img_size[1], 3))
        
        # Загрузка VGG16
        vgg_base = VGG16(
            weights='imagenet',
            include_top=False,
            input_shape=(self.img_size[0], self.img_size[1], 3)
        )
        vgg_base.trainable = False
        
        # Извлечение признаков для каждого кадра - VGG выдает 512 признаков после GlobalAveragePooling2D
        # Для VGG16 с include_top=False и input_shape=(224, 224, 3):
        # - Выходная форма: (None, 7, 7, 512)
        # - После GlobalAveragePooling2D: (None, 512)
        cnn_feature_extractor = Model(
            inputs=vgg_base.input,
            outputs=layers.GlobalAveragePooling2D()(vgg_base.output)
        )
        
        # Применение VGG к каждому кадру через TimeDistributed
        # Важно: TimeDistributed ожидает, что размер батча будет первым измерением
        time_distributed = layers.TimeDistributed(cnn_feature_extractor)(input_layer)
        
        # LSTM для временных зависимостей
        lstm_out = layers.LSTM(
            256,
            return_sequences=False,
            dropout=0.3,
            recurrent_dropout=0.3,
            #kernel_initializer='orthogonal'
        )(time_distributed)
        
        # Дополнительные слои
        x = layers.Dense(128, activation='relu')(lstm_out)
        x = layers.Dropout(0.5)(x)
        x = layers.BatchNormalization()(x)
        
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        x = layers.BatchNormalization()(x)
        
        # Выходной слой
        output_layer = layers.Dense(1, activation='sigmoid')(x)
        
        model = Model(inputs=input_layer, outputs=output_layer, name="Deepfake_Detector")
        
        return model
    
    def compile_model(self, learning_rate=0.0001):
        self.model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='binary_crossentropy',
            metrics=['accuracy', 
                    tf.keras.metrics.Precision(name='precision'),
                    tf.keras.metrics.Recall(name='recall'),
                    tf.keras.metrics.AUC(name='auc')]
        )
        print("Модель скомпилирована успешно!")

In [3]:
detector = DeepfakeVideoDetector()

In [4]:
detector.model.summary()

Model: "Deepfake_Detector"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 10, 224, 224, 3)  0         
                             ]                                   
                                                                 
 time_distributed (TimeDistr  (None, 10, 512)          14714688  
 ibuted)                                                         
                                                                 
 lstm (LSTM)                 (None, 256)               787456    
                                                                 
 dense (Dense)               (None, 128)               32896     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 batch_normalization (BatchN  (None, 128)        

In [5]:
class FrameSequenceGenerator(keras.utils.Sequence):
    def __init__(self, data_dir, batch_size=32, frames_per_sequence=FRAMES_PER_VIDEO, 
                 img_size=(HEIGHT, WIDTH), shuffle=True, is_train=True, overlap=OVERLAP, preprocess_function=None):
        self.data_dir = pathlib.Path(data_dir)
        self.batch_size = batch_size
        self.frames_per_sequence = frames_per_sequence
        self.img_size = img_size
        self.shuffle = shuffle
        self.is_train = is_train
        self.overlap = overlap
        self.preprocess_function=preprocess_function
        # Собираем все изображения из папок
        self.sequences = self._prepare_sequences()
        
        self.indexes = np.arange(len(self.sequences))
        if self.shuffle:
            np.random.shuffle(self.indexes)
        
        print(f"Found {len(self.sequences)} sequences")
        print(f"Real: {len([s for s in self.sequences if s['label'] == 0])}")
        print(f"Fake: {len([s for s in self.sequences if s['label'] == 1])}")
    
    def _prepare_sequences(self):
        sequences = []
        
        # Обрабатываем реальные кадры
        real_dir = self.data_dir / 'real'
        if real_dir.exists():
            real_images = sorted(list(real_dir.glob('*.jpg')) + 
                               list(real_dir.glob('*.png')) +
                               list(real_dir.glob('*.jpeg')))
            print(f"Found {len(real_images)} real images")
            
            # Разбиваем на последовательности с перекрытием
            if len(real_images) >= self.frames_per_sequence:
                step = self.frames_per_sequence - self.overlap
                for i in range(0, len(real_images) - self.frames_per_sequence + 1, max(1, step)):
                    sequence_frames = real_images[i:i + self.frames_per_sequence]
                    sequences.append({
                        'frames': sequence_frames,
                        'label': 0  # real
                    })
        else:
            print(f"Warning: Real directory {real_dir} does not exist!")
        
        # Обрабатываем фейковые кадры
        fake_dir = self.data_dir / 'fake'
        if fake_dir.exists():
            fake_images = sorted(list(fake_dir.glob('*.jpg')) + 
                               list(fake_dir.glob('*.png')) +
                               list(fake_dir.glob('*.jpeg')))
            print(f"Found {len(fake_images)} fake images")
            
            # Разбиваем на последовательности с перекрытием
            if len(fake_images) >= self.frames_per_sequence:
                step = self.frames_per_sequence - self.overlap
                for i in range(0, len(fake_images) - self.frames_per_sequence + 1, max(1, step)):
                    sequence_frames = fake_images[i:i + self.frames_per_sequence]
                    sequences.append({
                        'frames': sequence_frames,
                        'label': 1  # fake
                    })
        else:
            print(f"Warning: Fake directory {fake_dir} does not exist!")
        
        return sequences
    
    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))
    
    def __getitem__(self, index):
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, len(self.sequences))
        
        batch_indexes = self.indexes[start_idx:end_idx]
        batch_sequences = [self.sequences[i] for i in batch_indexes]
        
        X = np.zeros((len(batch_sequences), self.frames_per_sequence, 
                     self.img_size[0], self.img_size[1], 3), dtype=np.float32)
        y = np.zeros((len(batch_sequences), 1), dtype=np.float32)
        
        for i, seq in enumerate(batch_sequences):
            # Загружаем кадры
            for j, frame_path in enumerate(seq['frames']):
                try:
                    img = cv2.imread(str(frame_path))
                    if img is not None and self.preprocess_function == None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, (self.img_size[1], self.img_size[0]))
                        X[i, j] = img / 255.0
                    elif self.preprocess_function != None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, (self.img_size[1], self.img_size[0]))
                        img = self.preprocess_function(img)
                    else:
                        print(f"Warning: Could not load image {frame_path}")
                        # Заполняем нулями или предыдущим кадром
                        if j > 0:
                            X[i, j] = X[i, j-1]
                except Exception as e:
                    print(f"Error loading {frame_path}: {e}")
                    if j > 0:
                        X[i, j] = X[i, j-1]
            
            y[i] = seq['label']
        
        return X, y
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [6]:
    train_generator = FrameSequenceGenerator(
        data_dir='../balanced_data/train',
        batch_size=4,  # Уменьшил batch_size для экономии памяти
        frames_per_sequence=FRAMES_PER_VIDEO,
        img_size=(HEIGHT, WIDTH),
        shuffle=True,
        is_train=True,
        overlap=OVERLAP,
        preprocess_function=tf.keras.applications.vgg16.preprocess_input
    )
    
    # Для тестирования
    test_generator = FrameSequenceGenerator(
        data_dir='../balanced_data/test',
        batch_size=4,
        frames_per_sequence=FRAMES_PER_VIDEO,
        img_size=(HEIGHT, WIDTH),
        shuffle=True,
        is_train=False,
        overlap=0,  # Без перекрытия для теста
        preprocess_function=tf.keras.applications.vgg16.preprocess_input
    )

Found 15777 real images
Found 16000 fake images
Found 3177 sequences
Real: 1577
Fake: 1600
Found 4469 real images
Found 4037 fake images
Found 849 sequences
Real: 446
Fake: 403


In [7]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
#    keras.callbacks.ModelCheckpoint(
#        filepath='best_deepfake_model.keras',
#        monitor='val_accuracy',
#        save_best_only=True,
#        mode='max'
#    ),
#    keras.callbacks.CSVLogger('training_log.csv')
]

In [8]:
detector.compile_model()

Модель скомпилирована успешно!


In [9]:
print("\nStarting training...")
history = detector.model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    callbacks=callbacks,
    validation_steps=32,
    steps_per_epoch=256
)


Starting training...
Epoch 1/20
256/256 [==============================] - 146s 490ms/step - loss: 0.8861 - accuracy: 0.4961 - precision: 0.4728 - recall: 0.4806 - auc: 0.4955 - val_loss: 0.6982 - val_accuracy: 0.4922 - val_precision: 0.4922 - val_recall: 1.0000 - val_auc: 0.5000 - lr: 1.0000e-04
Epoch 2/20
256/256 [==============================] - 129s 503ms/step - loss: 0.8088 - accuracy: 0.5269 - precision: 0.5244 - recall: 0.5089 - auc: 0.5289 - val_loss: 0.7033 - val_accuracy: 0.5312 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_auc: 0.5000 - lr: 1.0000e-04
Epoch 3/20
256/256 [==============================] - 124s 484ms/step - loss: 0.8658 - accuracy: 0.4936 - precision: 0.4840 - recall: 0.4830 - auc: 0.4894 - val_loss: 0.6953 - val_accuracy: 0.4844 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_auc: 0.5000 - lr: 1.0000e-04
Epoch 4/20
256/256 [==============================] - 123s 481ms/step - loss: 0.8348 - accuracy: 0.4971 - precision: 0.5126 - recal